# BƯỚC 6 – Khai phá luật kết hợp (Apriori / FP-Growth)
**Đề tài 5 – Bệnh viện: Dự đoán tái nhập viện trong 30 ngày**

**Câu hỏi khám phá phụ:** Những mẫu sử dụng thuốc nào thường đồng xuất hiện, và mẫu nào có liên hệ thống kê với nhãn tái nhập viện dưới 30 ngày?

- Đơn vị giao dịch: **một lượt nhập viện** (`encounter_id`), không phải một bệnh nhân.
- Đây là **khai phá mang tính mô tả**, không khẳng định thuốc gây tái nhập viện hay sử dụng được để ra quyết định điều trị.
- Chỉ số: **support**, **confidence**, **lift**; đối chiếu thêm với tỷ lệ tái nhập viện chung (~11,16%).
- Không dùng kết quả từ tập test của Bước 4 để lựa chọn đặc trưng/siêu tham số cho mô hình chính. Kết quả toàn bộ dữ liệu ở đây chỉ dùng cho báo cáo mô tả.


## 0. Cài đặt thư viện (chỉ chạy một lần)
Trong PowerShell đã kích hoạt `.venv`:
```powershell
python -m pip install mlxtend
```
Nếu gặp lỗi do Smart App Control chặn DLL, kiểm tra `python -c "from mlxtend.frequent_patterns import fpgrowth; print('OK')"` trong PowerShell trước, không vô hiệu hóa bảo vệ Windows một cách tùy tiện.

In [ ]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
from IPython.display import display

root = Path.cwd()
if root.name.lower() == 'notebooks':
    root = root.parent
elif not (root/'data'/'processed'/'step3_cleaned.csv').exists():
    raise FileNotFoundError('Hãy mở JupyterLab từ D:\\Readmission_Project và lưu notebook trong notebooks/')
src = root/'data'/'processed'/'step3_cleaned.csv'
out = root/'reports'/'association_rules'
out.mkdir(parents=True, exist_ok=True)
print('Dữ liệu:', src)
print('Thư mục kết quả:', out)

## 1. Đọc dữ liệu và kiểm tra nhãn

In [ ]:
df = pd.read_csv(src, low_memory=False)
assert {'encounter_id','patient_nbr','readmitted_30d'}.issubset(df.columns)
assert df['encounter_id'].is_unique, 'encounter_id phải duy nhất'
assert df['readmitted_30d'].isin([0,1]).all(), 'Nhãn phải là 0/1'
base_rate = float(df['readmitted_30d'].mean())
print('Kích thước:', df.shape)
print('Số giao dịch:', len(df), '| bệnh nhân:', df.patient_nbr.nunique())
print(f'Tỷ lệ tái nhập viện chung: {base_rate:.2%}')
display(df[['encounter_id','patient_nbr','readmitted_30d']].head())

## 2. Chọn item cho giỏ giao dịch

Mỗi lượt nhập viện được mã hóa thành các item Boolean (có/không):
- `thuoc:<tên>` nếu thuốc được ghi là `Steady`, `Up` hoặc `Down` (khác `No`).
- `co_thay_doi_thuoc` nếu `change=Ch`.
- `co_dung_thuoc_tieu_duong` nếu `diabetesMed=Yes`.
- `tai_nhap_duoi_30_ngay` nếu nhãn bằng 1.

**Lưu ý:** `insulin=Up/Down` không có nghĩa người bệnh tự dùng insulin tại nhà; đây là trạng thái được ghi trong dữ liệu. Các item phản ánh cùng một lượt nhập viện. Không mã hóa `patient_nbr`, `encounter_id` hay nhãn `khong_tai_nhap` thành item.

In [ ]:
drug_candidates = [
    'metformin','repaglinide','nateglinide','chlorpropamide','glimepiride',
    'glipizide','glyburide','pioglitazone','rosiglitazone','acarbose',
    'insulin','glyburide-metformin'
]
missing_drugs = [c for c in drug_candidates if c not in df.columns]
assert not missing_drugs, f'Thiếu cột thuốc: {missing_drugs}'
transactions = pd.DataFrame(index=df.index)
for drug in drug_candidates:
    s = df[drug].astype('string').str.strip()
    transactions['thuoc:' + drug] = s.isin(['Steady','Up','Down']).fillna(False).astype(bool)
if 'change' in df:
    transactions['co_thay_doi_thuoc'] = df['change'].astype('string').str.strip().eq('Ch').fillna(False).astype(bool)
if 'diabetesMed' in df:
    transactions['co_dung_thuoc_tieu_duong'] = df['diabetesMed'].astype('string').str.strip().eq('Yes').fillna(False).astype(bool)
transactions['tai_nhap_duoi_30_ngay'] = df['readmitted_30d'].eq(1).astype(bool)
assert transactions.notna().sum().sum() == transactions.size
assert transactions['tai_nhap_duoi_30_ngay'].sum() == df['readmitted_30d'].sum()
frequencies = (transactions.mean()*100).sort_values(ascending=False).rename('ty_le_giao_dich_pct').reset_index().rename(columns={'index':'item'})
display(frequencies)
frequencies.to_csv(out/'item_frequencies.csv', index=False, encoding='utf-8-sig')

## 3. Chạy FP-Growth và Apriori
Mặc định `min_support=0.02` (ít nhất 2% lượt nhập viện) và giới hạn itemset tối đa 3 item để tránh bùng nổ tổ hợp. Có thể giảm xuống `0.01` sau khi xem kết quả, nhưng **ghi rõ lý do** trong nhật ký quyết định.

In [ ]:
MIN_SUPPORT = 0.02
MIN_CONFIDENCE = 0.15
MIN_LIFT = 1.0
MAX_LEN = 3

t0 = time.perf_counter()
fi_fp = fpgrowth(transactions, min_support=MIN_SUPPORT, use_colnames=True, max_len=MAX_LEN)
fp_seconds = time.perf_counter()-t0
t0 = time.perf_counter()
fi_ap = apriori(transactions, min_support=MIN_SUPPORT, use_colnames=True, max_len=MAX_LEN, low_memory=True)
ap_seconds = time.perf_counter()-t0
print(f'FP-Growth: {len(fi_fp)} itemsets ({fp_seconds:.2f}s)')
print(f'Apriori:   {len(fi_ap)} itemsets ({ap_seconds:.2f}s)')

# So sánh tần suất của các tập mục, không phụ thuộc thứ tự lưu
def normalise_fi(frame):
    return {tuple(sorted(s)): float(v) for s,v in zip(frame['itemsets'],frame['support'])}
a,b=normalise_fi(fi_fp), normalise_fi(fi_ap)
assert set(a)==set(b), 'Apriori và FP-Growth khác danh sách itemsets'
assert all(abs(a[k]-b[k])<1e-10 for k in a), 'Support hai thuật toán khác nhau'
print('ĐỐI CHIẾU APRIORI/FP-GROWTH: ĐẠT')

## 4. Luật kết hợp và luật liên quan đến tái nhập viện

In [ ]:
if fi_fp.empty:
    rules = pd.DataFrame()
else:
    rules = association_rules(fi_fp, metric='confidence', min_threshold=MIN_CONFIDENCE)
    rules = rules.loc[(rules['lift'] >= MIN_LIFT) & (rules['antecedents'].map(len)>=1)].copy()

def items_as_str(items):
    return ' & '.join(sorted(items))

if not rules.empty:
    rules['ve_trai'] = rules['antecedents'].map(items_as_str)
    rules['ve_phai'] = rules['consequents'].map(items_as_str)
    ordered = ['ve_trai','ve_phai','support','confidence','lift','leverage','conviction']
    rules_export = rules[ordered].sort_values(['lift','confidence','support'], ascending=False)
else:
    rules_export = pd.DataFrame(columns=['ve_trai','ve_phai','support','confidence','lift','leverage','conviction'])
rules_export.to_csv(out/'association_rules_all.csv', index=False, encoding='utf-8-sig')
print('Tổng luật sau lọc:',len(rules_export))
display(rules_export.head(20))

target_item = 'tai_nhap_duoi_30_ngay'
if not rules.empty:
    target_rules = rules.loc[rules['consequents'].map(lambda x: x==frozenset({target_item})) &
                             ~rules['antecedents'].map(lambda x: target_item in x)].copy()
    target_rules['ve_trai'] = target_rules['antecedents'].map(items_as_str)
    target_rules['ve_phai'] = target_item
    target_export = target_rules[['ve_trai','ve_phai','support','confidence','lift','leverage','conviction']].sort_values('lift', ascending=False)
else:
    target_export = pd.DataFrame(columns=rules_export.columns)
target_export.to_csv(out/'association_rules_readmission.csv',index=False,encoding='utf-8-sig')
print('Luật hướng tới tái nhập viện:', len(target_export))
if target_export.empty:
    print('Không có luật thỏa ngưỡng. Không được tự bịa luật; có thể thử MIN_SUPPORT=0.01 và ghi quyết định.')
else:
    display(target_export.head(20))

## 5. Diễn giải đúng Support – Confidence – Lift

Với luật `A → tái nhập dưới 30 ngày`:
- **Support** = tỷ lệ lượt nhập viện cùng có A và tái nhập dưới 30 ngày.
- **Confidence** = tỷ lệ tái nhập dưới 30 ngày trong các lượt có A.
- **Lift** = confidence chia cho tỷ lệ tái nhập chung; lớn hơn 1 biểu thị đồng xuất hiện cao hơn kỳ vọng độc lập.

**Cảnh báo:** Không suy ra tác dụng của thuốc, không suy ra quan hệ nhân quả, và không dùng luật này để khuyến nghị điều trị. Nhãn phản ánh lượt khám; cùng bệnh nhân có thể đóng góp nhiều lượt.

In [ ]:
summary = pd.DataFrame([{
    'n_luot_nhap_vien': len(df),
    'n_benh_nhan': df.patient_nbr.nunique(),
    'ty_le_tai_nhap_chung': base_rate,
    'min_support': MIN_SUPPORT,
    'min_confidence': MIN_CONFIDENCE,
    'min_lift': MIN_LIFT,
    'max_len': MAX_LEN,
    'so_itemset': len(fi_fp),
    'so_luat': len(rules_export),
    'so_luat_den_tai_nhap': len(target_export),
    'fp_growth_seconds': round(fp_seconds,3),
    'apriori_seconds': round(ap_seconds,3),
}])
display(summary)
summary.to_csv(out/'mining_summary.csv',index=False,encoding='utf-8-sig')
fi_csv=fi_fp.copy()
fi_csv['itemsets']=fi_csv['itemsets'].map(items_as_str)
fi_csv.sort_values('support',ascending=False).to_csv(out/'frequent_itemsets.csv',index=False,encoding='utf-8-sig')

fig,ax=plt.subplots(figsize=(10,5))
top=frequencies.head(12).sort_values('ty_le_giao_dich_pct')
ax.barh(top['item'], top['ty_le_giao_dich_pct'])
ax.set_xlabel('Tỷ lệ giao dịch (%)')
ax.set_title('Tần suất các item (12 item phổ biến nhất)')
fig.tight_layout()
fig.savefig(out/'01_item_frequency.png',dpi=160)
plt.show()

## 6. Nhật ký quyết định và kiểm tra file đầu ra

In [ ]:
log = pd.DataFrame([
 {'quyet_dinh':'Một giao dịch = một lượt nhập viện','ly_do':'Đúng đơn vị phân tích của dataset','anh_huong':'support/confidence tính theo lượt khám, không theo bệnh nhân'},
 {'quyet_dinh':'Dùng 12 cột thuốc đã chọn, thêm change/diabetesMed','ly_do':'Giới hạn số item, giữ ngữ cảnh điều trị','anh_huong':'Không đại diện mọi loại thuốc hiếm'},
 {'quyet_dinh':'Thuốc dùng nếu trạng thái Steady/Up/Down','ly_do':'Phân biệt có dùng với No','anh_huong':'Không phân biệt mức tăng/giảm liều'},
 {'quyet_dinh':'Thêm item nhãn tái nhập dưới 30 ngày','ly_do':'Khảo sát luật liên quan đến kết cục','anh_huong':'Luật mô tả, không suy luận nhân quả'},
 {'quyet_dinh':f'min_support={MIN_SUPPORT}, min_confidence={MIN_CONFIDENCE}, min_lift={MIN_LIFT}, max_len={MAX_LEN}', 'ly_do':'Kiểm soát độ phổ biến và số tổ hợp, ưu tiên luật có lift >=1','anh_huong':'Có thể bỏ sót luật hiếm; minh bạch ngưỡng'},
 {'quyet_dinh':'Dùng toàn bộ dữ liệu đã làm sạch cho khai phá mô tả','ly_do':'Mô tả tổng thể dữ liệu, không đánh giá dự báo','anh_huong':'Không dùng luật này để chọn mô hình/siêu tham số trên test của Bước 4'},
])
log.to_csv(out/'decision_log_step6.csv',index=False,encoding='utf-8-sig')
assert (out/'association_rules_all.csv').exists()
assert (out/'decision_log_step6.csv').exists()
print('Đã lưu thư mục:',out)
print('Kết quả:',len(fi_fp),'itemsets,',len(rules_export),'luật,',len(target_export),'luật liên quan nhãn')
print('HOÀN THÀNH BƯỚC 6 (phần thực thi).')

## 7. Đưa vào báo cáo

Trình bày: đơn vị giao dịch; cách định nghĩa item; ngưỡng lựa chọn; số itemset và số luật; 3–5 luật có ý nghĩa (nếu có); support/confidence/lift; so sánh Apriori và FP-Growth; hạn chế và đạo đức.

**Tập tin xuất:** `reports/association_rules/` gồm `item_frequencies.csv`, `frequent_itemsets.csv`, `association_rules_all.csv`, `association_rules_readmission.csv`, `mining_summary.csv`, `decision_log_step6.csv`, `01_item_frequency.png`.

**Bước 7:** Phân lớp nhị phân (Decision Tree so với baseline và ensemble), dùng bộ train/validation/test đã tách theo bệnh nhân ở Bước 4.